In [1]:
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath('../src'))
from data_loader import load_data, clean_data, calculate_metrics
import hypothesis_tests as ht

df = load_data('../data/insurance_data.csv')
df = clean_data(df)
df = calculate_metrics(df)

# Prepare KPI columns
# Claim Frequency: 1 if they claimed, 0 if not
df['IsClaimed'] = df['TotalClaims'] > 0

In [2]:
results = []

# 1. Risk across Provinces (Claim Frequency)
# Let's compare the two most frequent provinces in your data
top_provinces = df['Province'].value_counts().index[:2]
subset_prov = df[df['Province'].isin(top_provinces)]
_, p_prov = ht.perform_chi2_test(subset_prov, 'Province', 'IsClaimed')
results.append(['Provinces', 'Claim Frequency', 'Chi-Squared', p_prov])

# 2. Risk between ZipCodes (Claim Frequency)
top_zips = df['ZipCode'].value_counts().index[:2]
subset_zip = df[df['ZipCode'].isin(top_zips)]
_, p_zip = ht.perform_chi2_test(subset_zip, 'ZipCode', 'IsClaimed')
results.append(['ZipCodes', 'Claim Frequency', 'Chi-Squared', p_zip])

# 3. Margin difference between ZipCodes (Numerical)
zip_a_margin = df[df['ZipCode'] == top_zips[0]]['Margin']
zip_b_margin = df[df['ZipCode'] == top_zips[1]]['Margin']
_, p_margin = ht.perform_ttest(zip_a_margin, zip_b_margin)
results.append(['ZipCodes', 'Margin (Profit)', 'T-test', p_margin])

# 4. Risk difference between Women and Men (Claim Severity)
# Note: Severity is only for those who actually had a claim
male_sev = df[(df['Gender'] == 'Male') & (df['IsClaimed'] == True)]['TotalClaims']
female_sev = df[(df['Gender'] == 'Female') & (df['IsClaimed'] == True)]['TotalClaims']
_, p_gender = ht.perform_ttest(male_sev, female_sev)
results.append(['Gender', 'Claim Severity', 'T-test', p_gender])

# Create Results Table
results_df = pd.DataFrame(results, columns=['Feature', 'KPI', 'Test Used', 'P-Value'])
results_df['Decision'] = results_df['P-Value'].apply(lambda x: "Reject H0" if x < 0.05 else "Fail to Reject H0")
results_df

,Feature,KPI,Test Used,P-Value,Decision
0,Provinces,Claim Frequency,Chi-Squared,0.813885,Fail to Reject H0
1,ZipCodes,Claim Frequency,Chi-Squared,0.222601,Fail to Reject H0
2,ZipCodes,Margin (Profit),T-test,0.264234,Fail to Reject H0
3,Gender,Claim Severity,T-test,0.996406,Fail to Reject H0


Statistical Interpretation & Recommendations
Gender: With a p-value of 0.99, we fail to reject 
H0. There is no statistical evidence that claim severity differs by gender. Recommendation: Maintain gender-neutral pricing to avoid unnecessary complexity.
Geography (Province/ZipCode): Both frequency and margin tests returned p-values > 0.05. Recommendation: Current data suggests risk is distributed uniformly across the sampled locations. We should investigate other drivers like vehicle age or driver risk scores in the modeling phase.